In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold,cross_val_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import ExtraTreesClassifier

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

In [3]:
train=pd.read_csv(r"F:\playground-series-s6e6\train.csv")
test=pd.read_csv(r"F:\playground-series-s6e6\test.csv")

In [4]:
train.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [5]:
test.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
2,577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
3,577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
4,577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


In [6]:
print(train.shape)
print(test.shape)

(577347, 12)
(247435, 11)


In [7]:
TARGET="class"
ID_COL="id"

train_ids=train[ID_COL]
test_ids=test[ID_COL]

X=train.drop(columns=[TARGET, ID_COL])
y=train[TARGET]
X_test=test.drop(columns=[ID_COL])

In [8]:
target_encoder=LabelEncoder()
y_encoded=target_encoder.fit_transform(y)

In [9]:
def feature_set_2(df):

    df = df.copy()
    # feature set 1
    df["u_g"]=df["u"] - df["g"]
    df["g_r"]=df["g"] - df["r"]
    df["r_i"]=df["r"] - df["i"]
    df["i_z"]=df["i"] - df["z"]
    df["u_r"]=df["u"] - df["r"]
    df["u_i"]=df["u"] - df["i"]
    df["u_z"]=df["u"] - df["z"]
    df["g_i"]=df["g"] - df["i"]
    df["g_z"]=df["g"] - df["z"]
    df["r_z"]=df["r"] - df["z"]
     
    # Redshift transforms
    df["log_redshift"]=np.log1p(df["redshift"])
    df["sqrt_redshift"]=np.sqrt(np.clip(df["redshift"], 0, None))
    df["redshift_sq"]=(df["redshift"]**2)

    # Redshift × Magnitude
    df["redshift_u"]=(df["redshift"] * df["u"])
    df["redshift_g"]=(df["redshift"] * df["g"])
    df["redshift_r"]=(df["redshift"] * df["r"])
    df["redshift_i"]=(df["redshift"] * df["i"])
    df["redshift_z"]=(df["redshift"] * df["z"])

    # Redshift × Color Indices
    df["redshift_u_g"]=(df["redshift"] * df["u_g"])
    df["redshift_g_r"]=(df["redshift"] * df["g_r"])
    df["redshift_r_i"]=(df["redshift"] * df["r_i"])
    df["redshift_i_z"]=(df["redshift"] * df["i_z"])

    # Ratios
    eps=1e-6
    df["redshift_div_u"]=(df["redshift"] / (np.abs(df["u"]) + eps))
    df["redshift_div_g"]=(df["redshift"] / (np.abs(df["g"]) + eps))
    df["redshift_div_r"]=(df["redshift"] / (np.abs(df["r"]) + eps))
    df["redshift_div_i"]=(df["redshift"] / (np.abs(df["i"]) + eps))
    df["redshift_div_z"]=(df["redshift"] / (np.abs(df["z"]) + eps))
    
    return df

In [10]:
X_fs2=feature_set_2(X)
X_test_fs2=feature_set_2(X_test)

In [11]:
X_fs2.to_csv("X_fs2.csv", index=False)
X_test_fs2.to_csv("X_test_fs2.csv", index=False)

In [ ]:
cat_cols = ["spectral_type","galaxy_population"]

ohe = ColumnTransformer(transformers=[("cat",OneHotEncoder(handle_unknown="ignore"),cat_cols)],
    remainder="passthrough")

X_ohe=ohe.fit_transform(X_fs2)
X_test_ohe=ohe.transform(X_test_fs2)

print(X_fs2.columns.to_list())
print(X_test_fs2.columns.to_list())

['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_i', 'u_z', 'g_i', 'g_z', 'r_z', 'log_redshift', 'sqrt_redshift', 'redshift_sq', 'redshift_u', 'redshift_g', 'redshift_r', 'redshift_i', 'redshift_z', 'redshift_u_g', 'redshift_g_r', 'redshift_r_i', 'redshift_i_z', 'redshift_div_u', 'redshift_div_g', 'redshift_div_r', 'redshift_div_i', 'redshift_div_z']
['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'u_i', 'u_z', 'g_i', 'g_z', 'r_z', 'log_redshift', 'sqrt_redshift', 'redshift_sq', 'redshift_u', 'redshift_g', 'redshift_r', 'redshift_i', 'redshift_z', 'redshift_u_g', 'redshift_g_r', 'redshift_r_i', 'redshift_i_z', 'redshift_div_u', 'redshift_div_g', 'redshift_div_r', 'redshift_div_i', 'redshift_div_z']


In [13]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42)

In [ ]:
X_mi=pd.get_dummies(X_fs2,drop_first=True)

mi_scores = mutual_info_classif(X_mi,y_encoded,random_state=42)

mi_df=pd.DataFrame({"Feature": X_mi.columns,"MI": mi_scores})
mi_df.sort_values("MI",ascending=False).head(50)

,Feature,MI
30,redshift_div_u,0.525467
31,redshift_div_g,0.523478
32,redshift_div_r,0.516958
7,redshift,0.514990
18,log_redshift,0.514987
19,sqrt_redshift,0.514810
20,redshift_sq,0.514671
33,redshift_div_i,0.512374
25,redshift_z,0.511830
34,redshift_div_z,0.509393


In [ ]:
et=ExtraTreesClassifier(n_estimators=500,random_state=42,n_jobs=-1)
et.fit(X_mi,y)
et_df=pd.DataFrame({
    "Feature": X_mi.columns,
    "Importance": et.feature_importances_
})
et_df.sort_values("Importance",ascending=False).head(50)

,Feature,Importance
36,spectral_type_M,0.101092
38,galaxy_population_Red_Sequence,0.066619
19,sqrt_redshift,0.049699
18,log_redshift,0.048707
7,redshift,0.038084
33,redshift_div_i,0.037834
31,redshift_div_g,0.033819
32,redshift_div_r,0.033252
23,redshift_r,0.032764
34,redshift_div_z,0.032315


In [ ]:
lgbm_imp_model=LGBMClassifier(random_state=42,verbose=-1)
lgbm_imp_model.fit(X_ohe,y)
imp_df = pd.DataFrame({
    "Feature": ohe.get_feature_names_out(),
    "Importance": lgbm_imp_model.feature_importances_
})
imp_df.sort_values("Importance",ascending=False).head(50)

,Feature,Importance
7,remainder__delta,1572
6,remainder__alpha,1557
12,remainder__z,433
22,remainder__g_z,401
8,remainder__u,373
9,remainder__g,365
14,remainder__u_g,318
10,remainder__r,291
11,remainder__i,279
36,remainder__redshift_div_u,267


In [ ]:
lgbm_imp = LGBMClassifier(
    subsample=1.0,
    reg_lambda=0.1,
    reg_alpha=1,
    num_leaves=127,
    n_estimators=1000,
    min_child_samples=10,
    max_depth=-1,
    learning_rate=0.03,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

lgbm_imp.fit(X_ohe,y)
imp_df=pd.DataFrame({"Feature": ohe.get_feature_names_out(),"Importance": lgbm_imp.feature_importances_})
top_features=imp_df.sort_values("Importance",ascending=False).head(30)
print(top_features)

                      Feature  Importance
6            remainder__alpha       39863
7            remainder__delta       39424
23             remainder__r_z       14979
9                remainder__g       14777
16             remainder__r_i       14520
14             remainder__u_g       14428
8                remainder__u       14275
17             remainder__i_z       14239
12               remainder__z       14153
21             remainder__g_i       14135
22             remainder__g_z       13972
10               remainder__r       13966
15             remainder__g_r       13346
18             remainder__u_r       13187
11               remainder__i       13046
33    remainder__redshift_g_r       12185
34    remainder__redshift_r_i       12105
35    remainder__redshift_i_z       12024
32    remainder__redshift_u_g       12011
20             remainder__u_z       11606
19             remainder__u_i       11467
36  remainder__redshift_div_u        4991
27      remainder__redshift_u     

In [18]:
xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=len(np.unique(y_encoded)),
    eval_metric="mlogloss",
    n_estimators=700,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    colsample_bylevel=1.0,
    colsample_bynode=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    gamma=0.0,
    min_child_weight=1,
    max_delta_step=0,
    grow_policy="depthwise",
    max_leaves=0,
    max_bin=256,
    scale_pos_weight=1,
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
scores = cross_val_score(xgb,X_ohe,y_encoded,cv=cv,scoring="balanced_accuracy")
print("Mean CV:",scores.mean())
print("Std CV :",scores.std())

Mean CV: 0.9560521630722623
Std CV : 0.0006519022365564494


In [19]:
lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=len(np.unique(y_encoded)),
    boosting_type="gbdt",
    n_estimators=900,
    learning_rate=0.01,
    num_leaves=127,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_samples=20,
    min_child_weight=1e-3,
    max_bin=255,
    device="gpu",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)
lgbm_scores=cross_val_score(lgbm,X_ohe,y_encoded,cv=cv,scoring="balanced_accuracy",n_jobs=-1)
print("Mean CV :", lgbm_scores.mean())
print("Std CV  :", lgbm_scores.std())

Mean CV : 0.956495559008242
Std CV  : 0.0005933676272478061


In [21]:
lgbm.fit(X_ohe, y_encoded)
test_predictions = lgbm.predict(X_test_ohe)
test_predictions = target_encoder.inverse_transform(test_predictions)

In [ ]:
submission_df = pd.DataFrame({"id": test_ids,"class": test_predictions })
submission_df.to_csv("submission.csv", index=False)
print(submission_df.head())

Submission file successfully created and saved as 'submission.csv'!
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY


In [ ]:
lgbm2 = LGBMClassifier(
    objective="multiclass",
    num_class=len(np.unique(y_encoded)),
    boosting_type="gbdt",
    n_estimators=900,
    learning_rate=0.1,
    num_leaves=127,
    max_depth=-1,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_samples=20,
    min_child_weight=1e-3,
    max_bin=255,
    device="gpu",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)
lgbm_scores2=cross_val_score(lgbm2,X_ohe,y_encoded,cv=cv,scoring="balanced_accuracy",n_jobs=-1)
print("Mean CV :", lgbm_scores2.mean())
print("Std CV  :", lgbm_scores2.std())

Mean CV : 0.955937737059448
Std CV  : 0.0006199744582456523


In [ ]:
lgbm2.fit(X_ohe, y_encoded)
test_predictions = lgbm2.predict(X_test_ohe)
test_predictions = target_encoder.inverse_transform(test_predictions)

submission_df = pd.DataFrame({"id": test_ids,"class": test_predictions})
submission_df.to_csv("submission__tt.csv", index=False)
print("Submission file successfully created")
print(submission_df.head())

Submission file successfully created and saved as 'submission.csv'!
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY


In [ ]:
xgb.fit(X_ohe, y_encoded)
test_predictions = xgb.predict(X_test_ohe)
test_predictions = target_encoder.inverse_transform(test_predictions)

submission_df = pd.DataFrame({"id": test_ids,"class": test_predictions})
submission_df.to_csv("submission__tt.csv", index=False)
print("Submission file successfully created")
print(submission_df.head())

Submission file successfully created and saved as 'submission.csv'!
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY


In [ ]:
xgb2 = XGBClassifier(
    objective="multi:softprob",
    num_class=len(np.unique(y_encoded)),
    eval_metric="mlogloss",
    n_estimators=900,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    colsample_bylevel=1.0,
    colsample_bynode=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    gamma=0.0,
    min_child_weight=1,
    max_delta_step=0,
    grow_policy="depthwise",
    max_leaves=0,
    max_bin=256,
    scale_pos_weight=1,
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
scores2 = cross_val_score(xgb2,X_ohe,y_encoded,cv=cv,scoring="balanced_accuracy")
print("Mean CV:",scores2.mean())
print("Std CV :",scores2.std())

Mean CV: 0.9558786151985265
Std CV : 0.0006750910605014193


In [ ]:
xgb2.fit(X_ohe, y_encoded)
test_predictions = xgb2.predict(X_test_ohe)
test_predictions = target_encoder.inverse_transform(test_predictions)

submission_df = pd.DataFrame({"id": test_ids,"class": test_predictions})
submission_df.to_csv("submission__tt3.csv", index=False)
print("Submission file successfully created")
print(submission_df.head())

Submission file successfully created and saved as 'submission.csv'!
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY
